# Capítulo 1 — Importação de dados (Data Import)

**Livro:** começa na página **28** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py (modelos via OpenRouter) e corpus.py (base de conhecimento de exemplo)
from ia import *
from corpus import DOCS
import re, json, math, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO, "| documentos na base:", len(DOCS))

## Reading simple text with a DataLoader · PDF p. 30

#### 🧪 Exercício: Um carregador de documentos

Escreva `carregar_txt(caminho)` que devolve um dicionário `{"texto": ..., "metadados": {"fonte", "tamanho"}}`. Grave dois arquivos .txt temporários com textos da base `DOCS` e carregue os dois.

In [ ]:
import tempfile, os
def carregar_txt(caminho):
    with open(caminho, encoding="utf-8") as f:
        t = f.read()
    return {"texto": t, "metadados": {"fonte": os.path.basename(caminho), "tamanho": len(t)}}
pasta = tempfile.mkdtemp()
for d in DOCS[:2]:
    with open(os.path.join(pasta, d["id"] + ".txt"), "w", encoding="utf-8") as f:
        f.write(d["texto"])
docs = [carregar_txt(os.path.join(pasta, n)) for n in sorted(os.listdir(pasta))]
print([d["metadados"] for d in docs])

## Parsing specific elements with the JSON loader · PDF p. 42

#### 🧪 Exercício: Extraindo campos de JSON

Converta `DOCS` para uma string JSON e, de volta, extraia só os documentos atualizados a partir de 2025, guardando `id` e `titulo` como metadados e `texto` como conteúdo (como faz um *JSON loader* com `jq_schema`).

In [ ]:
bruto = json.dumps(DOCS, ensure_ascii=False)
docs = [{"conteudo": d["texto"], "metadados": {"id": d["id"], "titulo": d["titulo"]}}
        for d in json.loads(bruto) if d["data"] >= "2025-01-01"]
print(len(docs), [d["metadados"]["titulo"] for d in docs])

## Importing table data in CSV format · PDF p. 53

#### 🧪 Exercício: Uma linha de CSV = um documento

Monte um CSV (em memória, com `io.StringIO`) com colunas `servico, preco, observacao` para impressão e cópias. Transforme cada linha num documento de texto do tipo "servico: ...; preco: ...".

In [ ]:
import csv, io
csv_txt = "servico,preco,observacao\nimpressao pb,0.30,por pagina\nimpressao colorida,1.50,por pagina\ncopia obra rara,,nao permitida\n"
docs = ["; ".join(f"{k}: {v or '-'}" for k, v in linha.items()) for linha in csv.DictReader(io.StringIO(csv_txt))]
print("\n".join(docs))

## Crawling and parsing web documents · PDF p. 57

#### 🧪 Exercício: Limpando HTML para indexar

Com `html.parser`, extraia de um HTML de exemplo só o texto útil (títulos e parágrafos), ignorando `<script>`, `<nav>` e `<footer>`.

In [ ]:
from html.parser import HTMLParser
html = "<nav>Menu Início</nav><h1>Eventos</h1><p>Clube de leitura na 1ª quinta.</p><script>track()</script><p>Contação aos sábados.</p><footer>© Aurora</footer>"
class Limpador(HTMLParser):
    IGNORAR = {"script", "style", "nav", "footer"}
    def __init__(self):
        super().__init__(); self.pilha, self.partes = [], []
    def handle_starttag(self, tag, attrs): self.pilha.append(tag)
    def handle_endtag(self, tag):
        if self.pilha: self.pilha.pop()
    def handle_data(self, data):
        if data.strip() and not (set(self.pilha) & self.IGNORAR): self.partes.append(data.strip())
l = Limpador(); l.feed(html); print(" | ".join(l.partes))

## Markdown file titles and structure · PDF p. 63

#### 🧪 Exercício: Dividindo Markdown pelos títulos

Escreva um Markdown com `#` e `##` sobre a biblioteca e divida em seções, guardando em cada pedaço o caminho dos títulos (por ex. `Serviços > Empréstimo`).

In [ ]:
md_txt = """# Serviços
Texto geral.
## Empréstimo
Até 5 livros por 14 dias.
## Impressão
R$ 0,30 por página.
# Espaços
## Sala de estudos
40 lugares."""
secoes, caminho = [], {}
for linha in md_txt.splitlines():
    m = re.match(r"(#+)\s+(.*)", linha)
    if m:
        nivel = len(m.group(1)); caminho = {k: v for k, v in caminho.items() if k < nivel}; caminho[nivel] = m.group(2)
        secoes.append({"titulos": " > ".join(caminho[k] for k in sorted(caminho)), "texto": ""})
    elif linha.strip():
        secoes[-1]["texto"] += linha.strip() + " "
for s in secoes: print(s)